## Methodology

The task involves recognizing and reconstructing text sequences from highly distorted grayscale images. The images contain various real-world challenges such as background noise, overlapping characters, blur, shape deformation, occlusion, and irregular spacing.

To solve this, we adopted a **CRNN (Convolutional Recurrent Neural Network)** architecture combined with **CTC (Connectionist Temporal Classification)** loss. This approach is well-suited for sequence recognition tasks as it does not require explicit character segmentation. 

The CNN extracts visual features from the input image, while the BiLSTM captures sequential dependencies across the feature map. CTC loss enables the model to learn alignment between the visual features and the target character sequence without needing pre aligned labels. During inference, CTC greedy decoding is used to convert the model's output into the final text string.

## Model Architecture

We used a custom **CRNN** model consisting of two main components:

- **CNN Feature Extractor**: A 5-layer convolutional network that progressively downsamples the input image (256×64) while increasing feature depth. It uses BatchNorm and ReLU activations after each convolution, followed by MaxPooling layers (with asymmetric strides) to reduce spatial dimensions while preserving width for sequence modeling.

- **Sequence Modeling**: A 2-layer **Bidirectional LSTM** with 256 hidden units per direction. The LSTM processes the feature sequence of length 128 and hidden size 512 (after concatenation of forward and backward passes).

- **Output Layer**: A fully connected layer that maps the LSTM output to the character vocabulary size (including the CTC blank token).

The model takes a grayscale image as input and outputs a probability distribution over characters at each time step, which is then decoded using CTC.

## Training Strategy

- **Loss Function**: `nn.CTCLoss` with `blank=0` and `zero_infinity=True`.
- **Optimizer**: Adam optimizer with an initial learning rate of 0.0005.
- **Learning Rate Scheduler**: `ReduceLROnPlateau` (factor=0.5, patience=3) that reduces learning rate when validation loss plateaus.
- **Gradient Clipping**: Applied with `max_norm=5.0` to stabilize training.
- **Data Preprocessing**: All images were resized to 256×64 pixels and normalized using mean and standard deviation of 0.5.
- **Data Split**: 90% of the data was used for training and 10% was held out as a validation set.
- **Training Duration**: The model was trained for 50 epochs with a batch size of 64.
- **Device**: Training was performed on CPU due to MPS (Apple Silicon) incompatibility with CTCLoss.
- **Model Checkpointing**: The model with the lowest validation Character Error Rate (CER) was saved as `crnn_model_best.pth`.

## Outputs & Results

### Training Outputs:
- Training and validation loss curves showed consistent decrease over epochs.
- Validation CER improved significantly
- Sample predictions during training showed progressive improvement in both length and accuracy of predicted sequences.

### Final Deliverables:
- **`crnn_model_best.pth`**: Best performing model weights based on validation CER.
- **`submission.csv`**: Contains predictions for all test images in the required format:
  ```csv
  image,prediction
  test-0.png,AXU323
  test-1.png,BT91KD
  ...
### Character Error Rate (CER) obtained is 0.02%

In [9]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from torchmetrics.text import CharErrorRate
from glob import glob

In [20]:
def build_vocab(csv_path):
    df = pd.read_csv(csv_path)
    all_labels = df['text'].astype(str).tolist()
    unique_chars = set()
    for label in all_labels:
        for char in label:
            unique_chars.add(char)
    sorted_chars = sorted(list(unique_chars))
    char_to_int = {'-': 0}
    int_to_char = {0: '-'}
    for idx, char in enumerate(sorted_chars, start=1):
        char_to_int[char] = idx
        int_to_char[idx] = char
    return char_to_int, int_to_char, df

class OCRDataset(Dataset):
    def __init__(self, dataframe, img_dir, char_to_int, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.char_to_int = char_to_int
        self.transform = transform
        self.img_width = 256
        self.img_height = 64

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = str(self.dataframe.iloc[idx]['image'])
        text_label = str(self.dataframe.iloc[idx]['text'])
        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Could not read: {img_path}")
        image = cv2.resize(image, (self.img_width, self.img_height))
        image = np.expand_dims(image, axis=-1)
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        target_seq = [self.char_to_int[char] for char in text_label]
        target_tensor = torch.tensor(target_seq, dtype=torch.long)
        target_length = torch.tensor(len(target_seq), dtype=torch.long)
        return image, target_tensor, target_length

def ctc_collate_fn(batch):
    images, targets, target_lengths = zip(*batch)
    images = torch.stack(images, 0)
    targets = torch.cat(targets, 0)
    target_lengths = torch.stack(target_lengths, 0)
    return images, targets, target_lengths




### Model Architecture

In [ ]:
class CustomCRNN(nn.Module):
    def __init__(self, num_chars):
        super(CustomCRNN, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=(2, 1)),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(4, 1), stride=(4, 1))
        )
        self.rnn = nn.LSTM(
            input_size=512, hidden_size=256, num_layers=2,
            bidirectional=True, batch_first=True, dropout=0.25
        )
        self.fc = nn.Linear(512, num_chars)

    def forward(self, x):
        conv_out = self.cnn(x)
        conv_out = conv_out.squeeze(2)
        conv_out = conv_out.permute(0, 2, 1)
        rnn_out, _ = self.rnn(conv_out)
        output = self.fc(rnn_out)
        return output
    


In [4]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, num_epochs):
    model.train()
    epoch_loss = 0.0
    for batch_idx, (images, targets, target_lengths) in enumerate(train_loader):
        images = images.to(device)
        targets = targets.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()
        outputs = model(images)                          
        log_probs = F.log_softmax(outputs, dim=2).permute(1, 0, 2)  
        
        batch_size = images.size(0)
        time_steps = log_probs.size(0)
        input_lengths = torch.full((batch_size,), time_steps, dtype=torch.long, device=device)

        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        epoch_loss += loss.item()
        if batch_idx % 50 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    return epoch_loss / len(train_loader)
def ctc_greedy_decode(pred_indices, int_to_char, blank=0):
    """Standard CTC greedy decoding (collapse repeats then remove blanks)"""
    collapsed = []
    prev = -1
    for p in pred_indices:
        if p != prev:
            collapsed.append(p)
        prev = p
    text = ''.join([int_to_char.get(c, '') for c in collapsed if c != blank])
    return text

def evaluate_val_set(model, val_df, img_dir, char_to_int, int_to_char, device, blank=0):
    """Evaluate CER on validation set + show samples"""
    model.eval()
    cer_metric = CharErrorRate()
    predictions, ground_truths = [], []

    with torch.no_grad():
        for idx in range(len(val_df)):
            img_name = str(val_df.iloc[idx]['image'])
            true_text = str(val_df.iloc[idx]['text'])
            img_path = os.path.join(img_dir, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue
            image = cv2.resize(image, (256, 64))
            image = np.expand_dims(image, axis=-1)
            image = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5], std=[0.5])
            ])(image).unsqueeze(0).to(device)

            outputs = model(image)
            log_probs = F.log_softmax(outputs, dim=2)
            pred_indices = log_probs.argmax(dim=2).squeeze(0).cpu().numpy()
            pred_text = ctc_greedy_decode(pred_indices, int_to_char, blank)

            predictions.append(pred_text)
            ground_truths.append(true_text)

    if predictions:
        cer = cer_metric(predictions, ground_truths)
        print(f"\n>>> Validation CER: {cer:.4f} ({cer*100:.2f}%)")
        print("Sample predictions (first 8):")
        for i in range(min(8, len(predictions))):
            status = "✅" if predictions[i] == ground_truths[i] else "❌"
            print(f"  {ground_truths[i]:8s} | {predictions[i]:8s} | {status}")
        return cer
    return None

### Execution 

In [ ]:
if __name__ == "__main__":
    CSV_PATH = 'train-labels.csv'
    IMG_DIR  = '/Users/kanishka/Documents/cig_ps/train_images'   
    BATCH_SIZE = 64
    EPOCHS = 30
    VAL_EVAL_EVERY = 5

    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        print("⚠️  MPS (Apple Silicon) detected.")
        print("   CTCLoss is NOT supported on MPS in current PyTorch.")
        print("   → Falling back to CPU (recommended for stability).\n")
        device = torch.device('cpu')
    else:
        device = torch.device('cpu')

    print(f"Final training device: {device}\n")
    
    print("Building vocabulary from CSV...")
    char_to_int, int_to_char, df = build_vocab(CSV_PATH)
    VOCAB_SIZE = len(char_to_int)
    print(f"Vocabulary complete. Total unique characters + 1 Blank Token: {VOCAB_SIZE}\n")

    print("Validating dataset files...")
    df['image'] = df['image'].astype(str).str.strip()
    valid_rows = [os.path.exists(os.path.join(IMG_DIR, img_name)) for img_name in df['image']]
    df = df[valid_rows]
    print(f"Found {len(df)} valid images.\n")

    print("Splitting dataset into Training (90%) and Validation (10%)...")
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)
    val_df.to_csv('validation_set.csv', index=False)
    print(f"==> Training on {len(train_df)} images | Validation on {len(val_df)} images\n")

    train_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])

    train_dataset = OCRDataset(train_df, IMG_DIR, char_to_int, transform=train_transforms)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=ctc_collate_fn, num_workers=0)

    print("Loading Custom CRNN Architecture...")
    model = CustomCRNN(num_chars=VOCAB_SIZE).to(device)
    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters\n")

    criterion = nn.CTCLoss(blank=0, zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=0.0005)            
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    print("Starting Training Loop...\n")
    best_cer = float('inf')

    for epoch in range(EPOCHS):
        avg_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, EPOCHS)
        print(f"==> End of Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")
        scheduler.step(avg_loss)

    
        if (epoch + 1) % VAL_EVAL_EVERY == 0 or epoch == EPOCHS - 1:
            cer = evaluate_val_set(model, val_df, IMG_DIR, char_to_int, int_to_char, device)
            if cer is not None and cer < best_cer:
                best_cer = cer
                torch.save(model.state_dict(), 'crnn_model_best.pth')
                print("  Saved new best model (crnn_model_best.pth)\n")

    print("\nTraining Complete! Saving final weights...")
    torch.save(model.state_dict(), 'crnn_model_final.pth')
    print("Model saved as 'crnn_model_final.pth' and 'crnn_model_best.pth'")

### Inference

In [14]:
TEST_IMG_DIR = '/Users/kanishka/Documents/cig_ps/test_images'
MODEL_PATH = 'crnn_model_best.pth'
OUTPUT_CSV = 'submission.csv'

device = torch.device('cpu')

In [15]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()
print(f"✅ Best model loaded from: {MODEL_PATH}")

✅ Best model loaded from: crnn_model_best.pth


In [16]:
def predict_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if image is None:
        return ""
    image = cv2.resize(image, (256, 64))
    image = np.expand_dims(image, axis=-1)
    image = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        log_probs = F.log_softmax(outputs, dim=2)
        pred_indices = log_probs.argmax(dim=2).squeeze(0).cpu().numpy()
        pred_text = ctc_greedy_decode(pred_indices, int_to_char)   
    return pred_text

In [18]:
print("\nRunning inference on test images...")

test_images = sorted(glob(os.path.join(TEST_IMG_DIR, "*.png")), 
                     key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split('-')[-1]))

results = []

for img_path in test_images:
    filename = os.path.basename(img_path)
    prediction = predict_image(img_path)
    results.append({"image": filename, "prediction": prediction})



Running inference on test images...


In [19]:
submission_df = pd.DataFrame(results)
submission_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Submission file saved as: {OUTPUT_CSV}")
print(f"Total test images: {len(results)}")
print("\nPreview:")
print(submission_df.head(10))


✅ Submission file saved as: submission.csv
Total test images: 5000

Preview:
        image prediction
0  test-0.png     QVTQ8A
1  test-1.png     7PSW9D
2  test-2.png     WJ2WNY
3  test-3.png     RFHJD4
4  test-4.png     K7ZUF2
5  test-5.png     CPMUBK
6  test-6.png     UZDRAW
7  test-7.png     2YDPJR
8  test-8.png     H5SG63
9  test-9.png     B2Z823
